In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.svm import SVR 
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# Load dataset
dataset = pd.read_excel("HousePricePrediction.xlsx")

print(dataset.head(5))
print("Dataset shape:", dataset.shape)

# Data type analysis
obj = (dataset.dtypes == 'object')
object_cols = list(obj[obj].index)
print("Categorical variables:", len(object_cols))

int_ = (dataset.dtypes == 'int')
num_cols = list(int_[int_].index)
print("Integer variables:", len(num_cols))  

fl = (dataset.dtypes == 'float') 
fl_cols = list(fl[fl].index)
print("Float variables:", len(fl_cols)) 

# Correlation heatmap for numerical features
numerical_dataset = dataset.select_dtypes(include=['number'])
plt.figure(figsize=(12, 6))  
sns.heatmap(numerical_dataset.corr(),
            cmap='BrBG',
            fmt='.2f',
            linewidths=2,
            annot=True)
plt.title('Correlation Heatmap of Numerical Features')
plt.tight_layout()
plt.show() 

# Plot unique values for categorical features
unique_values = []
for col in object_cols:
    unique_values.append(dataset[col].unique().size)
plt.figure(figsize=(10, 6))
plt.title('No. unique values of Categorical Features')
plt.xticks(rotation=90)  
sns.barplot(x=object_cols, y=unique_values)
plt.tight_layout()
plt.show()

# Plot distribution of categorical features
plt.figure(figsize=(18, 36))
plt.title('Categorical Features: Distribution')
plt.xticks(rotation=90)
index = 1

for col in object_cols:
    y = dataset[col].value_counts()
    plt.subplot(11, 4, index) 
    plt.xticks(rotation=90)
    sns.barplot(x=list(y.index), y=y)
    index += 1
plt.tight_layout()  
plt.show()

# Data preprocessing
dataset.drop(['Id'], axis=1, inplace=True, errors='ignore')
dataset['SalePrice'] = dataset['SalePrice'].fillna(dataset['SalePrice'].mean())
new_dataset = dataset.dropna()

print("Missing values after cleaning:", new_dataset.isnull().sum().sum())

# One-hot encoding
s = (new_dataset.dtypes == 'object')
object_cols = list(s[s].index)
print("Categorical Variables:")
print(object_cols)
print('No. of. categorical features: ', len(object_cols))

OH_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
OH_cols = pd.DataFrame(OH_encoder.fit_transform(new_dataset[object_cols]))
OH_cols.index = new_dataset.index
OH_cols.columns = OH_encoder.get_feature_names_out() 
df_final = new_dataset.drop(object_cols, axis=1)
df_final = pd.concat([df_final, OH_cols], axis=1)

# Prepare features and target
X = df_final.drop(['SalePrice'], axis=1)
Y = df_final['SalePrice']

# Split data
X_train, X_valid, Y_train, Y_valid = train_test_split(
    X, Y, train_size=0.8, test_size=0.2, random_state=0)

# Model training and evaluation
print("\n" + "="*50)
print("BASE MODELS")
print("="*50)

# SVM Regression - Fixed: using SVR instead of SVC
model_SVR = SVR()
model_SVR.fit(X_train, Y_train)  
Y_pred = model_SVR.predict(X_valid)
print("SVM MAPE:", mean_absolute_percentage_error(Y_valid, Y_pred))

# Random Forest
model_RFR = RandomForestRegressor(n_estimators=10, random_state=0)  
model_RFR.fit(X_train, Y_train)
Y_pred = model_RFR.predict(X_valid)
print("Random Forest MAPE:", mean_absolute_percentage_error(Y_valid, Y_pred))

# Linear Regression
model_LR = LinearRegression()
model_LR.fit(X_train, Y_train)
Y_pred = model_LR.predict(X_valid)
print("Linear Regression MAPE:", mean_absolute_percentage_error(Y_valid, Y_pred))

# Decision Tree (added for comparison)
model_DT = DecisionTreeRegressor(random_state=0)
model_DT.fit(X_train, Y_train)
Y_pred = model_DT.predict(X_valid)
print("Decision Tree MAPE:", mean_absolute_percentage_error(Y_valid, Y_pred))

# ENSEMBLE METHODS
print("\n" + "="*50)
print("ENSEMBLE METHODS")
print("="*50)

# Bagging with Decision Trees
bagging_dt = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=0),
    n_estimators=50,
    random_state=0
)
bagging_dt.fit(X_train, Y_train)
Y_pred_bag_dt = bagging_dt.predict(X_valid)
print(f"Bagging with Decision Trees MAPE: {mean_absolute_percentage_error(Y_valid, Y_pred_bag_dt):.4f}")

# Gradient Boosting
gradient_boost = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=0
)
gradient_boost.fit(X_train, Y_train)
Y_pred_gb = gradient_boost.predict(X_valid)
print(f"Gradient Boosting MAPE: {mean_absolute_percentage_error(Y_valid, Y_pred_gb):.4f}")

# AdaBoost
adaboost = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=4, random_state=0),
    n_estimators=50,
    random_state=0,
    learning_rate=0.1
)
adaboost.fit(X_train, Y_train)
Y_pred_ada = adaboost.predict(X_valid)
print(f"AdaBoost MAPE: {mean_absolute_percentage_error(Y_valid, Y_pred_ada):.4f}")

# Additional metrics for better evaluation
print("\n" + "="*50)
print("ADDITIONAL METRICS")
print("="*50)

# Calculate MAE for all models
models = {
    'SVM': model_SVR,
    'Random Forest': model_RFR,
    'Linear Regression': model_LR,
    'Decision Tree': model_DT,
    'Bagging': bagging_dt,
    'Gradient Boosting': gradient_boost,
    'AdaBoost': adaboost
}

for name, model in models.items():
    Y_pred = model.predict(X_valid)
    mape = mean_absolute_percentage_error(Y_valid, Y_pred)
    mae = mean_absolute_error(Y_valid, Y_pred)
    print(f"{name}: MAPE = {mape:.4f}, MAE = {mae:.2f}")

print("\nModel training and evaluation completed successfully!")